# Track 1 · 12 Alljoined prepare — windows and targets for `track1_11_reve_alljoined`

**One job.** Write what `track1_11` trains on, from Alljoined-1.6M (`nm000134`, 32-channel
Emotiv, the sealed montage):

1. per person, harness-style EEG windows (notch, 0.1-75 Hz band-pass, 120 Hz, per-recording
   robust scaling, epoch -0.2 to 0.8 s, baseline over the first 0.2 s, clamp 20), one
   `windows/sub-XX_train.npz` and one `windows/sub-XX_test.npz` each;
2. the frozen DINOv2-giant target table, one 1536-d vector per stimulus image;
3. `manifest.json` (sampling rate) beside them.

Everything lands in `ROOT/Track1/alljoined/`. Raw recordings are cached in `ROOT/Track1/raw`
(shared with track1_01, so nothing already downloaded is fetched twice). Self-contained.

Split (Alljoined's own, from track1_01): runs 05-19 present training images, runs 01-04 the
200 fixed test images, and the two sets do not intersect. The sealed gallery is larger than the
200-image local one.

## The target space is a fixed specification (unchanged from track1_01)

```yaml
target: {name: HuggingFaceImage, model_name: facebook/dinov2-giant, layers: 0.6667,
         token_aggregation: mean, imsize: 518}
brain_model_output_size: 1536
```
`layers: 0.6667` = hidden state 26 of 41; the self-test below asserts it before the long pass.

## What is unverified (first thing to read in the output)

- how many windows each run gives, and how many images the selection needs to embed (printed
  before the long steps);
- whether the EDF channel names resolve to REVE's electrode names (the step raises if not);
- that `events.tsv` onsets line up with the EDF clock (the `sample` column is compared to
  `onset` and a warning is printed if they disagree).

The embedding pass is resumable (512-image shards), and each person's windows are skipped if
already written, so a dropped session costs little.


## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'   # tracks go in ROOT/Track1, ROOT/Track2, ...

N_SUBJECTS = 20            # Alljoined-1.6M has 20 people
TRAIN_RUNS_PER_PERSON = 15 # runs 05-19, rotated over the person's sessions; 'all' takes every one
TEST_RUNS_PER_PERSON = 4   # runs 01-04 (the 200 fixed test images), one session

MODEL_NAME = 'facebook/dinov2-giant'   # the scored target space; do not change to fit
DEPTH = 0.6667                         # layers, from the track config
IMSIZE = 518
EMBED_BATCH = 16           # images per forward pass
FETCH_WORKERS = 8          # concurrent downloads (16 made the server reset connections)
MAX_FAILURES = 100         # stop the embedding pass after this many images fail for good
MAX_DEAD_BATCHES = 5       # ... or after this many batches in a row where nothing downloaded
SHARD = 512                # embeddings per resumable shard

# Harness preprocessing (neuralbench defaults + image task), as in track1_08.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
TMIN, DURATION = -0.2, 1.0
BASELINE = (-0.2, 0.0)
CLAMP = 20.0

print(f'root       {ROOT}')
print(f'people     {N_SUBJECTS}, {TRAIN_RUNS_PER_PERSON} train runs + {TEST_RUNS_PER_PERSON} test runs each')
print(f'target     {MODEL_NAME} @ depth {DEPTH}, mean tokens, imsize {IMSIZE}')


In [ ]:
%pip install -q 'eegdash>=0.9.1' mne transformers torchvision pillow requests

import importlib
for mod in ('eegdash', 'mne', 'transformers', 'torchvision', 'PIL', 'requests', 'torch'):
    m = importlib.import_module(mod)
    print(f'{mod:14} {getattr(m, "__version__", "ok")}')

### Helpers

Everything Track 1 needs, inline. The repo carries fuller versions of these in `src/`, but
keeping them here is what lets this notebook run as a single uploaded file.

In [ ]:
from pathlib import Path

DATASET = 'nm000134'              # Alljoined-1.6M on NeMAR
SLUG = 'Track1'                   # folder under ROOT
EXCLUDE_SESSIONS = ('02old',)     # superseded recordings
EXPECT_SFREQ, EXPECT_NCHANS = 256.0, 32

# Alljoined splits its stimuli by run: runs 01-04 present the 200 fixed test images,
# runs 05-19 present training images, and the two sets do not intersect. The run number
# *is* the cross-stimulus split.
TRAIN_RUNS = tuple(f'{i:02d}' for i in range(5, 20))
TEST_RUNS = tuple(f'{i:02d}' for i in range(1, 5))

BYTES_PER_SAMPLE = 4              # float32


def paths(root):
    base = Path(root) / SLUG
    out = {'base': base, 'raw': base / 'raw', 'prepared': base / 'prepared',
           'weights': base / 'weights', 'submissions': base / 'submissions'}
    for v in out.values():
        v.mkdir(parents=True, exist_ok=True)
    out['manifest'] = base / 'manifest.json'
    return out


def estimate_mb(record):
    """Uncompressed size of one recording, from metadata alone. The EDFs are ~half this."""
    return (record.get('ntimes') or 0) * (record.get('nchans') or 0) * BYTES_PER_SAMPLE / 1e6


def targets_mb(n_stimuli, dim=1536):
    return n_stimuli * dim * BYTES_PER_SAMPLE / 1e6


def select_run_split(records, n_subjects=12, runs_per_subject=1,
                     exclude_sessions=EXCLUDE_SESSIONS):
    """Split by run id, taking both sides of the split from one session per subject.

    Both sides come from the same session on purpose. Track 1's shift is cross-stimulus,
    so holding subject and session fixed keeps a local score about images rather than
    about session drift. Which session a subject contributes rotates with its index, so a
    one-run-per-subject selection is not silently all ses-01.

    Deterministic, so re-running picks the same recordings and a resumed run still matches.
    """
    exclude = set(exclude_sessions)
    grouped = {}
    for r in records:
        if r.get('_has_missing_files') or r.get('session') in exclude:
            continue
        grouped.setdefault(r['subject'], {}).setdefault(r['session'], []).append(r)

    train, test = [], []
    for i, subject in enumerate(sorted(grouped)[:n_subjects]):
        sessions = sorted(grouped[subject])
        order = sessions[i % len(sessions):] + sessions[:i % len(sessions)]
        for session in order:
            pool = grouped[subject][session]
            tr = sorted((r for r in pool if r['run'] in TRAIN_RUNS), key=lambda r: r['run'])
            te = sorted((r for r in pool if r['run'] in TEST_RUNS), key=lambda r: r['run'])
            if len(tr) < runs_per_subject or len(te) < runs_per_subject:
                continue
            tr = tr[i % len(tr):] + tr[:i % len(tr)]   # vary which images each subject brings
            train += tr[:runs_per_subject]
            test += te[:runs_per_subject]
            break
    return train, test


def select_people(records, n_subjects=20, train_runs='all', test_runs=4,
                  exclude_sessions=EXCLUDE_SESSIONS):
    """Per person: up to `train_runs` runs from 05-19 and `test_runs` from 01-04. Test runs
    come from the person's first usable session, train runs fill from that session onward,
    so the image sets stay disjoint (runs 01-04 and 05-19 never share an image).
    Deterministic: the same call always picks the same recordings."""
    exclude = set(exclude_sessions)
    grouped = {}
    for r in records:
        if r.get('_has_missing_files') or r.get('session') in exclude:
            continue
        grouped.setdefault(r['subject'], {}).setdefault(r['session'], []).append(r)
    train, test = [], []
    for i, subject in enumerate(sorted(grouped)[:n_subjects]):
        sessions = sorted(grouped[subject])
        order = sessions[i % len(sessions):] + sessions[:i % len(sessions)]
        te = []
        for session in order:
            te = sorted((r for r in grouped[subject][session] if r['run'] in TEST_RUNS),
                        key=lambda r: r['run'])
            if len(te) >= test_runs:
                break
        test += te[:test_runs]
        tr = []
        for session in order:
            tr += sorted((r for r in grouped[subject][session] if r['run'] in TRAIN_RUNS),
                         key=lambda r: r['run'])
        train += tr if train_runs == 'all' else tr[:train_runs]
    return train, test


def report(title, checks):
    """checks: list of (name, ok, detail)."""
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


print('helpers defined')

In [ ]:
import os

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
P = paths(ROOT)

# DINOv2-giant is 4.5 GB and is needed once. Keeping it on ephemeral /content costs a
# re-download on a new session and saves the Drive space for data that must persist.
os.environ['HF_HOME'] = '/content/hf'

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'dataset    {DATASET}')
print(f'raw        {P["raw"]}')
print(f'prepared   {P["prepared"]}')
print(f'device     {DEVICE}')
if DEVICE == 'cpu':
    print('\nNo GPU. dinov2-giant on CPU will take hours for this many images.')
    print('Switch the runtime to a GPU, or set MODEL_NAME to facebook/dinov2-base for a dry run.')

## Step 1 — choose the recordings

Selection is by run, not by byte budget, because the run number is the train/test split.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import mne
mne.set_log_level('ERROR')
from eegdash import EEGDash, EEGDashDataset

records = [r for r in EEGDash().find({'dataset': DATASET})
           if r['bids_relpath'].startswith('sub-')]
print(f'catalogue: {len(records)} records, {len({r["subject"] for r in records})} subjects')

train_recs, test_recs = select_people(
    records, n_subjects=N_SUBJECTS, train_runs=TRAIN_RUNS_PER_PERSON, test_runs=TEST_RUNS_PER_PERSON)

selected = train_recs + test_recs
split_of = {r['bids_relpath']: s for recs, s in ((train_recs, 'train'), (test_recs, 'test'))
            for r in recs}
per_person = {}
for r in selected:
    d = per_person.setdefault(r['subject'], {'train': 0, 'test': 0})
    d[split_of[r['bids_relpath']]] += 1
print('runs per person (train/test): ' + '  '.join(f'{s}:{d["train"]}/{d["test"]}' for s, d in sorted(per_person.items())))

sfreqs = {r.get('sampling_frequency') for r in selected}
nchans = {r.get('nchans') for r in selected}
report('selection:', [
    ('records found', len(selected) > 0, f'{len(selected)} records, ~{sum(estimate_mb(r) for r in selected):.0f} MB uncompressed'),
    ('sampling rate', sfreqs == {EXPECT_SFREQ}, f'{sfreqs} Hz'),
    ('channel count', nchans == {EXPECT_NCHANS}, f'{nchans}'),
    ('people', len(per_person) == N_SUBJECTS, f'{len(per_person)} distinct'),
    ('every person has test runs', all(d['test'] == TEST_RUNS_PER_PERSON for d in per_person.values()),
     'test runs per person'),
    ('excluded sessions absent', not ({r['session'] for r in selected} & set(EXCLUDE_SESSIONS)), f'{EXCLUDE_SESSIONS}'),
])

# Is it all on Drive already? Then there is nothing to do, and nothing is downloaded or embedded.
import numpy as np
OUT = P['base'] / 'alljoined'
expected = [OUT / 'windows' / f'sub-{s}_{sp}.npz'
            for s, sp in sorted({(r['subject'], split_of[r['bids_relpath']]) for r in selected})]
have = [f for f in expected if f.exists()]
print(f'windows already on Drive: {len(have)}/{len(expected)} files')
if len(have) == len(expected) and (OUT / 'image_embeddings.npz').exists() and (OUT / 'manifest.json').exists():
    win_keys = set()
    for f in expected:
        win_keys |= set(np.load(f)['keys'].tolist())
    gap = win_keys - set(np.load(OUT / 'image_embeddings.npz')['keys'].tolist())
    if not gap:
        raise SystemExit(f'Everything is already on Drive in {OUT} ({len(expected)} window files, '
                         f'{len(win_keys):,} images embedded). Nothing to do: run track1_11.')
    print(f'{len(gap):,} window images have no embedding yet; continuing')


## Step 2 — pull the EEG

Already-downloaded recordings are reused, so re-running this is free.

In [ ]:
import time

os.environ['EEGDASH_CACHE_DIR'] = str(P['raw'])
cached = sum((P['raw'] / DATASET / r['bids_relpath']).exists() for r in selected)
print(f'{cached}/{len(selected)} recordings already on Drive; only the rest are downloaded')
ds = EEGDashDataset(records=selected, cache_dir=P['raw'])

t0 = time.time()
fetched, failed = [], []
for i, d in enumerate(ds.datasets, 1):
    try:
        _ = d.raw                      # touching .raw is what downloads
        fetched.append(d)
    except Exception as e:
        failed.append((i, f'{type(e).__name__}: {e}'))
        print(f'  record {i} failed: {type(e).__name__}: {str(e)[:90]}')
print(f'{len(fetched)}/{len(selected)} recordings in {time.time() - t0:.0f}s')

on_disk_mb = sum(f.stat().st_size for f in P['raw'].rglob('*') if f.is_file()) / 1e6
print(f'{on_disk_mb:.0f} MB on Drive')

if fetched:
    raw = fetched[0].raw
    has_montage = raw.get_montage() is not None
    report('first recording:', [
        ('sampling rate', raw.info['sfreq'] == EXPECT_SFREQ, f'{raw.info["sfreq"]} Hz'),
        ('channel count', len(raw.ch_names) == EXPECT_NCHANS, f'{len(raw.ch_names)}'),
        ('annotations', len(raw.annotations) > 0, f'{len(raw.annotations)} events'),
        ('montage present', has_montage,
         'none, but REVE reads positions from channel names, not from the data'
         if not has_montage else 'yes'),
    ])

## Step 3 — read the stimulus lists

EEGDash caches each recording's BIDS `_events.tsv` next to the EDF, so the image shown at
every onset can be read directly.

The key is `stim_file`, not `value`. The dataset's own `events.json` records that `value` is
zero-padded to five digits in eleven sessions and is a placeholder for oddball rows in seven
more. `stim_file` names the JPG in every row that has one.

In [ ]:
import pandas as pd

STIM_KINDS = {'train': 'stim_train', 'test': 'stim_test'}
events, images = {}, {'train': set(), 'test': set()}

for rec in selected:
    rel = rec['bids_relpath']
    split = split_of[rel]
    tsv = P['raw'] / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    if not tsv.exists():
        print(f'  no events.tsv for {rel}')
        continue
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM_KINDS[split]].dropna(subset=['stim_file'])
    events[rel] = df[['onset', 'sample', 'stim_file']].reset_index(drop=True)
    images[split] |= set(df.stim_file)

n_ev = {s: sum(len(v) for k, v in events.items() if split_of[k] == s) for s in ('train', 'test')}
print(f'train   {n_ev["train"]:6,} events   {len(images["train"]):6,} unique images')
print(f'test    {n_ev["test"]:6,} events   {len(images["test"]):6,} unique images')

overlap = images['train'] & images['test']
print(f'\ntrain/test image overlap: {len(overlap)}   (must be 0 for a cross-stimulus split)')
assert not overlap, f'{len(overlap)} images appear in both splits'

all_images = sorted(images['train'] | images['test'])
print(f'to embed: {len(all_images):,} images  ->  {targets_mb(len(all_images)):.0f} MB table')
print(f'local gallery for scoring: {len(images["test"])} images, '
      f'chance top-5 {5 / max(len(images["test"]), 1):.2%}')

## Step 4 — build the target table

The model is loaded once and the images are streamed past it. Nothing is kept but the
1536-d vector.

The self-test is the guard against a silently wrong target space. It asserts the layer
index and the output dimension, which are the two things that would otherwise be wrong
without complaint. It runs before the long pass, so a mistake costs seconds.

In [ ]:
import io, requests, numpy as np, transformers
import logging
from urllib3.util.retry import Retry
logging.getLogger('urllib3').setLevel(logging.ERROR)   # one line per retry floods the output
from PIL import Image as PILImage
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel

STIM_URL = f'https://data.nemar.org/{DATASET}/v1.0.3/stimuli/{{}}'

_tf = transforms.Compose([transforms.Resize(IMSIZE), transforms.ToTensor()])
_parts = P['prepared'] / 'embedding_parts'
_done = set()
for _f in sorted(_parts.glob('part_*.npz')):
    _done |= set(np.load(_f)['keys'].tolist())
NEED_EMBED = any(n not in _done for n in all_images)
print(f'{len(_done):,} images already embedded; model {"needed" if NEED_EMBED else "not needed, not loaded"}')
if NEED_EMBED:
    _proc = AutoImageProcessor.from_pretrained(MODEL_NAME, do_rescale=False)
    _model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

_session = requests.Session()
# The default pool holds 10 connections; with more workers than that, requests
# discards and re-opens sockets for every batch.
_session.mount('https://', requests.adapters.HTTPAdapter(
    pool_connections=FETCH_WORKERS, pool_maxsize=FETCH_WORKERS,
    max_retries=Retry(total=4, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504))))


def fetch(name):
    """Stream one stimulus. The nemar URL 302-redirects to presigned S3."""
    r = _session.get(STIM_URL.format(name), timeout=(10, 40))
    r.raise_for_status()
    return PILImage.open(io.BytesIO(r.content)).convert('RGB')


def _safe_fetch(name):
    """Fetch, or None. Keeps one bad URL from ending a 12,000-image pass."""
    try:
        return fetch(name)
    except Exception:
        return None


def embed(pil_images):
    """The target space, exactly as neuralbench defines it. Returns (B, 1536) float32."""
    tens = [_tf(im).float() for im in pil_images]
    inputs = _proc(images=tens, return_tensors='pt')
    pv = inputs['pixel_values']
    if pv.isnan().any():                 # uniform images produce NaNs; neuralset zeroes them
        pv[pv.isnan()] = 0
        inputs['pixel_values'] = pv.float()
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        states = _model(**inputs, output_hidden_states=True).hidden_states
    if states is None:
        raise RuntimeError(f'{MODEL_NAME} returned no hidden states on transformers '
                           f'{transformers.__version__}; hidden-state collection regressed '
                           'for some encoders in transformers 5')
    stacked = torch.cat([s.unsqueeze(1) for s in states], dim=1)   # (B, layers, tokens, F)
    n_layers = stacked.shape[1]
    idx = int(DEPTH * (n_layers - 1))    # truncation, matching neuralset
    return stacked[:, idx].mean(dim=1).float().cpu().numpy(), idx, n_layers


if NEED_EMBED:
    # self-test on generated images: no network, so a server hiccup cannot fail it
    probe = [PILImage.fromarray(np.random.default_rng(i).integers(0, 255, (300, 400, 3), dtype=np.uint8)) for i in range(4)]
    vecs, layer_idx, n_layers = embed(probe)
    print(f'{MODEL_NAME}: {n_layers} hidden states, taking index {layer_idx}')
    print(f'embedding {vecs.shape}, norms {np.linalg.norm(vecs, axis=1).round(2)}')

    if MODEL_NAME == 'facebook/dinov2-giant':
        assert (n_layers, layer_idx) == (41, 26), (n_layers, layer_idx)
        assert vecs.shape[1] == 1536, vecs.shape
        print('matches the scored target space')
    else:
        print(f'DRY RUN on {MODEL_NAME}: this table is NOT the scored target space')
    del probe, vecs

In [ ]:
from concurrent.futures import ThreadPoolExecutor

parts = P['prepared'] / 'embedding_parts'
parts.mkdir(parents=True, exist_ok=True)

done = set()
for f in sorted(parts.glob('part_*.npz')):
    done |= set(np.load(f)['keys'].tolist())
todo = [n for n in all_images if n not in done]
print(f'{len(done):,} already embedded, {len(todo):,} to go')

pool = ThreadPoolExecutor(FETCH_WORKERS)
buf_keys, buf_vecs, failures = [], [], []
shard_no = len(list(parts.glob('part_*.npz')))
t0 = time.time()
dead, STOPPED = 0, False


def flush():
    global buf_keys, buf_vecs, shard_no
    if not buf_keys:
        return
    np.savez(parts / f'part_{shard_no:05d}.npz',
             keys=np.array(buf_keys), emb=np.stack(buf_vecs).astype('float32'))
    shard_no += 1
    buf_keys, buf_vecs = [], []


for start in range(0, len(todo), EMBED_BATCH):
    names = todo[start:start + EMBED_BATCH]
    got, imgs = [], []
    for name, im in pool.map(lambda n: (n, _safe_fetch(n)), names):
        if im is None:
            failures.append(name)
        else:
            got.append(name)
            imgs.append(im)
    dead = 0 if imgs else dead + 1
    if imgs:
        vecs, _, _ = embed(imgs)
        buf_keys += got
        buf_vecs += list(vecs)
    if len(buf_keys) >= SHARD:
        flush()
    if len(failures) >= MAX_FAILURES or dead >= MAX_DEAD_BATCHES:
        STOPPED = True
        break
    n_seen = start + len(names)
    if n_seen % (EMBED_BATCH * 20) == 0 or n_seen >= len(todo):
        rate = n_seen / max(time.time() - t0, 1e-9)
        print(f'  {n_seen:6,}/{len(todo):,}  {rate:5.1f} img/s  '
              f'eta {(len(todo) - n_seen) / max(rate, 1e-9) / 60:5.1f} min  '
              f'{len(failures)} failed')

flush()
if STOPPED:
    raise RuntimeError(f'Stopped early: {len(failures)} images failed and {dead} batches in a row '
                       'downloaded nothing, so the image server is not answering. Finished shards '
                       'are saved; run this cell again later (or lower FETCH_WORKERS).')
print(f'\ndone in {(time.time() - t0) / 60:.1f} min, {len(failures)} failures')

In [ ]:
# Retry anything that failed, once, serially. A failure here is usually a transient
# 5xx from the redirect, not a missing image.
if failures:
    print(f'retrying {len(failures)}')
    retry_keys, retry_vecs, still = [], [], []
    for name in failures:
        im = _safe_fetch(name)
        if im is None:
            still.append(name)
            continue
        v, _, _ = embed([im])
        retry_keys.append(name)
        retry_vecs.append(v[0])
    if retry_keys:
        np.savez(parts / f'part_{shard_no:05d}.npz',
                 keys=np.array(retry_keys), emb=np.stack(retry_vecs).astype('float32'))
        shard_no += 1
    failures = still
    print(f'{len(failures)} still failing: {failures[:10]}')
else:
    print('nothing to retry')

## Step 5 — consolidate and record what was written

In [ ]:
keys, mat = [], []
for f in sorted(parts.glob('part_*.npz')):
    z = np.load(f)
    keys += z['keys'].tolist()
    mat.append(z['emb'])

emb = np.concatenate(mat) if mat else np.zeros((0, 1536), 'float32')
order = {k: i for i, k in enumerate(keys)}          # last write wins on a duplicate
uniq = sorted(order)
emb = emb[[order[k] for k in uniq]]

table = P['prepared'] / 'image_embeddings.npz'
np.savez(table, keys=np.array(uniq), emb=emb, model=MODEL_NAME, depth=DEPTH,
         imsize=IMSIZE, token_aggregation='mean')
print(f'{table}  {emb.shape}  {table.stat().st_size / 1e6:.0f} MB')

missing = sorted(set(all_images) - set(uniq))
print(f'coverage: {len(uniq):,}/{len(all_images):,} images, {len(missing)} missing')
if missing:
    print(f'  missing: {missing[:10]}')

# Coverage says every image produced a row. It cannot say the rows are usable, so
# check the values too: a degenerate embedding would pass the count and poison the fit.
norms = np.linalg.norm(emb, axis=1)
report('table:', [
    ('shape', emb.shape[1] == (1536 if MODEL_NAME.endswith('giant') else emb.shape[1]),
     f'{emb.shape} {emb.dtype}'),
    ('no missing images', not missing, f'{len(missing)} missing'),
    ('finite values', bool(np.isfinite(emb).all()), 'no NaN or Inf'),
    ('distinct keys', len(uniq) == len(set(uniq)), f'{len(uniq):,} keys'),
    ('no zero rows', bool((norms > 0).all()), f'{int((norms == 0).sum())} all-zero'),
    ('norms', bool(norms.max() / max(norms.min(), 1e-9) < 10),
     f'min {norms.min():.2f}  median {np.median(norms):.2f}  max {norms.max():.2f}'),
])

In [ ]:
import json

meta = {
    'dataset': DATASET,
    'target': {'model': MODEL_NAME, 'layers': DEPTH, 'token_aggregation': 'mean',
               'imsize': IMSIZE, 'dim': int(emb.shape[1]),
               'source': 'neuralbench/tasks/eeg/image/config.yaml (neuralbench 0.3.1)'},
    'epoch': {'start_s': -0.2, 'duration_s': 1.0, 'baseline_s': [0.0, 0.2],
              'note': 'from the same config; ISI is ~200 ms so epochs overlap heavily'},
    'table': {'path': str(table), 'images': len(uniq), 'missing': missing},
    'recordings': {rel: {'split': split_of[rel],
                         'subject': rel.split('/')[0].removeprefix('sub-'),
                         'events': len(df),
                         'images': sorted(set(df.stim_file))}
                   for rel, df in events.items()},
    'counts': {'train_events': n_ev['train'], 'test_events': n_ev['test'],
               'train_images': len(images['train']), 'test_images': len(images['test'])},
}
(P['prepared'] / 'prepare_manifest.json').write_text(json.dumps(meta, indent=2))
print(f'manifest -> {P["prepared"] / "prepare_manifest.json"}')

used = sum(f.stat().st_size for f in P['base'].rglob('*') if f.is_file()) / 1e6
print(f'\nDrive used by {SLUG}: {used:.0f} MB')
print(f'  EEG        {on_disk_mb:6.0f} MB')
print(f'  embeddings {table.stat().st_size / 1e6:6.0f} MB')
print('\nNext: epoch the EEG at -0.2 to 0.8 s, run REVE over the windows, fit the ridge head.')
print('Note: no montage is set on these recordings. REVE does not need one: it looks')
print('up electrode positions by channel name in its own bank. Matters only for an')
print('encoder that reads positions out of the data.')

## Step 6 — the windows, per person

Harness preprocessing, as `track1_08` does for THINGS-EEG2: notch at 50/60 Hz and harmonics,
0.1-75 Hz band-pass, resample to 120 Hz, per-recording robust scaling (median and IQR per
channel), epoch -0.2 to 0.8 s around each stimulus onset, subtract the mean of the first 0.2 s,
clamp at 20. Written one file per person per split; people already written are skipped.

In [ ]:
import gc
OUT = P['base'] / 'alljoined'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
N_TIMES = int(round(DURATION * SFREQ))
PRE = int(round(-TMIN * SFREQ))
B0, B1 = (int(round((b - TMIN) * SFREQ)) for b in BASELINE)

# standard 10-10 names, so an EDF channel called 'Fp1', 'FP1' or 'EEG Fp1' resolves the same way
_std = {n.casefold(): n for n in mne.channels.make_standard_montage('standard_1020').ch_names}


def clean_names(names):
    out, bad = [], []
    for n in names:
        k = n.replace('EEG ', '').replace('EEG-', '').strip().casefold()
        if k in _std:
            out.append(_std[k])
        else:
            bad.append(n)
    if bad:
        raise RuntimeError(f'channel names that do not resolve to a standard 10-10 electrode: {bad}')
    return out


def epoch_recording(raw, onsets_s):
    """Harness-style windows around each onset. Returns (X float32 (n, ch, N_TIMES), kept mask)."""
    raw = raw.copy().pick('eeg')
    nyq = raw.info['sfreq'] / 2
    freqs = sorted({f for base in NOTCH for f in np.arange(base, min(nyq, 301), base) if f < nyq})
    raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    raw.resample(SFREQ, verbose=False)
    x = raw.get_data()
    med = np.median(x, axis=1, keepdims=True)
    iqr = np.subtract(*np.percentile(x, [75, 25], axis=1))[:, None]
    iqr[iqr == 0] = 1.0
    x = ((x - med) / iqr).astype('float32')
    s = np.round(np.asarray(onsets_s) * SFREQ).astype(int) - PRE
    ok = (s >= 0) & (s + N_TIMES <= x.shape[1])
    X = np.stack([x[:, i:i + N_TIMES] for i in s[ok]]) if ok.any() else np.zeros((0, x.shape[0], N_TIMES), 'float32')
    X = X - X[:, :, B0:B1].mean(axis=2, keepdims=True)
    return np.clip(X, -CLAMP, CLAMP), ok


by_person = {}
for rec in selected:
    by_person.setdefault((rec['subject'], split_of[rec['bids_relpath']]), []).append(rec)

CHANNELS, t0 = None, time.time()
for (subject, split), recs in sorted(by_person.items()):
    out = WIN / f'sub-{subject}_{split}.npz'
    if out.exists():
        print(f'  sub-{subject} {split}: already written, skipped')
        continue
    Xs, keys = [], []
    for rec in recs:
        rel = rec['bids_relpath']
        edf = P['raw'] / DATASET / rel
        ev = events.get(rel)
        if ev is None or not edf.exists():
            print(f'  {rel}: missing events or EDF, skipped')
            continue
        raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
        names = clean_names(raw.copy().pick('eeg').ch_names)
        if len(names) != EXPECT_NCHANS:
            raise RuntimeError(f'{rel}: {len(names)} EEG channels, expected {EXPECT_NCHANS}: {names}')
        if CHANNELS is None:
            CHANNELS = names
        elif names != CHANNELS:
            raise RuntimeError(f'{rel}: channel order differs from the first recording')
        gap = np.median(np.abs(ev['onset'].values * raw.info['sfreq'] - ev['sample'].values))
        if gap > 2:
            print(f'  WARNING {rel}: median |onset*sfreq - sample| = {gap:.1f} samples; check alignment')
        X, ok = epoch_recording(raw, ev['onset'].values)
        Xs.append(X.astype('float16')); keys += ev['stim_file'].values[ok].tolist()
        del raw, X
    X = np.concatenate(Xs)
    np.savez(out, X=X, keys=np.array(keys), subject=subject, split=split,
             session=','.join(sorted({r['session'] for r in recs})), channels=np.array(CHANNELS))
    print(f'  sub-{subject} {split}: {len(recs)} runs -> {len(X):,} windows, '
          f'{out.stat().st_size / 1e6:.0f} MB, {time.time() - t0:.0f}s')
    del X, Xs
    gc.collect()


In [ ]:
import shutil
shutil.copy(table, OUT / 'image_embeddings.npz')
(OUT / 'manifest.json').write_text(json.dumps({
    'sfreq': SFREQ, 'epoch': {'tmin': TMIN, 'duration': DURATION, 'baseline': BASELINE, 'clamp': CLAMP},
    'channels': CHANNELS, 'target': {'model': MODEL_NAME, 'depth': DEPTH, 'imsize': IMSIZE},
    'people': sorted({k[0] for k in by_person}), 'train_runs_per_person': TRAIN_RUNS_PER_PERSON,
    'test_runs_per_person': TEST_RUNS_PER_PERSON}, indent=2))

files = sorted(WIN.glob('*.npz'))
n_win = sum(len(np.load(f)['keys']) for f in files)
emb_keys = set(np.load(OUT / 'image_embeddings.npz')['keys'].tolist())
missing = set()
for f in files:
    missing |= set(np.load(f)['keys'].tolist()) - emb_keys
report('Alljoined windows:', [
    ('files', len(files) == 2 * N_SUBJECTS, f'{len(files)} files, {n_win:,} windows'),
    ('channels', CHANNELS is not None and len(CHANNELS) == EXPECT_NCHANS, f'{CHANNELS}'),
    ('every window has an embedding', not missing, f'{len(missing)} keys without one'),
])
print(f'\nNext: run track1_11_reve_alljoined on Colab (reads {OUT}).')
